# Roller Coaster Database EDA

A reproducible walkthrough based on Rob Mulla’s EDA workflow: understand, prepare, explore features, inspect relationships, and answer a question. The included `coaster_db.csv` is the dataset snapshot used for the report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('coaster_db.csv')
print('Shape:', df.shape)
display(df.head())
display(df.dtypes.value_counts())
print('Exact duplicate rows:', df.duplicated().sum())

## Prepare and assess data

Use normalized measurement columns. Keep missing measurements missing and let each analysis use available cases.

In [ ]:
cols = ['year_introduced', 'speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean']
num = df[cols].apply(pd.to_numeric, errors='coerce')
display(num.describe().T)
display(pd.DataFrame({'missing': num.isna().sum(), 'missing_pct': (num.isna().mean()*100).round(1)}))
display(df['Type_Main'].value_counts().to_frame('records'))

## Feature distributions and type comparison

In [ ]:
df['Type_Main'].value_counts().plot.bar(color=['#2563eb','#d97706','#6b7280'], title='Coasters by main type', ylabel='Records')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for kind in ['Steel', 'Wood', 'Other']:
    num.loc[df.Type_Main.eq(kind), 'speed_mph'].dropna().plot.hist(bins=25, alpha=.5, label=kind, ax=ax)
ax.set(title='Recorded maximum speed by coaster type', xlabel='Speed (mph)', ylabel='Rides'); ax.legend(); plt.tight_layout(); plt.show()

display(pd.concat([df.Type_Main, num.speed_mph], axis=1).groupby('Type_Main').speed_mph.agg(['count','median','mean']).round(2))

## Relationships and a focused question

Correlation is descriptive and does not show causation. Pairwise sample counts vary with missingness.

In [ ]:
metrics = ['speed_mph','height_ft','Inversions_clean','Gforce_clean']
display(num[metrics].corr().round(3))

fig, ax = plt.subplots(figsize=(7,5))
for kind in ['Steel','Wood','Other']:
    mask = df.Type_Main.eq(kind) & num.speed_mph.notna() & num.height_ft.notna()
    ax.scatter(num.loc[mask,'height_ft'], num.loc[mask,'speed_mph'], alpha=.5, s=18, label=kind)
ax.set(title='Height and speed', xlabel='Height (ft)', ylabel='Speed (mph)'); ax.legend(); plt.tight_layout(); plt.show()

num = num.assign(decade=(num.year_introduced // 10 * 10))
display(num.groupby('decade').agg(rides=('year_introduced','size'), median_speed=('speed_mph','median'), median_height=('height_ft','median')).tail(12).round(1))

## Interpretation

The supplied report summarizes the findings: height and speed have a strong positive association among the 156 rows with both measurements (*r* ≈ 0.83), while median speed is similar across broad coaster types. Sparse height data and uneven coverage constrain generalization. See `EDA_Report.md` for limitations and sources.